In [1]:
# Hyperparameter Tuning

In [2]:
# Rebuild Dataset + Pipeline (cleaned + engineered)

import numpy as np
import pandas as pd
import time

rng = np.random.default_rng(42)
n = 500

neighborhood = rng.choice(['Downtown', 'Suburb', 'Rural'], size=n, p=[0.4, 0.4, 0.2])
house_style = rng.choice(['1Story', '2Story', 'Split'], size=n)
sqft = rng.normal(1800, 500, n).clip(500, 4000)
bedrooms = rng.integers(1, 6, n)
age = rng.integers(0, 80, n)
distance_to_city = rng.normal(15, 8, n).clip(0.5, 60)

neighborhood_premium = {'Downtown': 1.4, 'Suburb': 1.1, 'Rural': 0.8}
style_premium = {'1Story': 1.0, '2Story': 1.15, 'Split': 1.05}
base_price = (sqft * 120 + bedrooms * 8000 - age * 500 - distance_to_city * 900)
price = base_price * np.array([neighborhood_premium[v] for v in neighborhood]) \
              * np.array([style_premium[v] for v in house_style])
price = price + rng.normal(0, 15000, n)

df = pd.DataFrame({
    'neighborhood': neighborhood, 'house_style': house_style,
    'sqft': sqft, 'bedrooms': bedrooms, 'age': age,
    'distance_to_city': distance_to_city, 'price': price
})

outlier_idx = rng.choice(n, size=12, replace=False)
df.loc[outlier_idx[:6], 'sqft'] = rng.uniform(7000, 9000, 6)
df.loc[outlier_idx[6:], 'distance_to_city'] = rng.uniform(150, 200, 6)
mask = df.index.isin(outlier_idx[:6])
df.loc[mask, 'price'] = df.loc[mask, 'sqft'] * 120 + rng.normal(0, 15000, mask.sum())

def cap_outliers(series, lower_q=0.01, upper_q=0.99):
    lower, upper = series.quantile(lower_q), series.quantile(upper_q)
    return series.clip(lower, upper)

df['sqft'] = cap_outliers(df['sqft'])
df['distance_to_city'] = cap_outliers(df['distance_to_city'])
df['sqft_per_bedroom'] = df['sqft'] / df['bedrooms']
df['is_far_commute'] = (df['distance_to_city'] > 25).astype(int)

numeric_cols = ['sqft', 'bedrooms', 'age', 'distance_to_city', 'sqft_per_bedroom', 'is_far_commute']
categorical_cols = ['neighborhood', 'house_style']

X = df[numeric_cols + categorical_cols]
y = df['price']

print(X.shape, y.shape)


(500, 8) (500,)


In [3]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV, cross_val_score

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])

pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Baseline (default hyperparameters) for comparison
baseline_scores = -cross_val_score(pipeline, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f"Baseline (default RF params) CV Mean RMSE: {baseline_scores.mean():.2f}  Std: {baseline_scores.std():.2f}")


Baseline (default RF params) CV Mean RMSE: 34531.26  Std: 11617.72


In [4]:
# Define param_grid — 4 Key Hyperparameters

param_grid_small = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}
# Small grid = 2*2*2*2 = 16 combinations x 5 folds = 80 fits -> feasible for GridSearchCV

param_dist_large = {
    'model__n_estimators': [50, 100, 150, 200, 300, 400],
    'model__max_depth': [None, 5, 10, 15, 20, 30],
    'model__min_samples_split': [2, 4, 6, 8, 10],
    'model__min_samples_leaf': [1, 2, 3, 4, 5],
    'model__max_features': ['sqrt', 'log2', None]
}
# Larger space = 6*6*5*5*3 = 2700 combinations -> exhaustive grid search infeasible,
# RandomizedSearchCV samples a fixed budget (n_iter) instead


In [5]:
# GridSearchCV — Exhaustive Search on Small Grid

start = time.time()
grid_search = GridSearchCV(
    pipeline, param_grid_small, cv=kf,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid_search.fit(X, y)
grid_time = time.time() - start

print(f"GridSearchCV runtime: {grid_time:.1f}s")
print(f"Combinations tried: {len(grid_search.cv_results_['params'])}")
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV RMSE: {-grid_search.best_score_:.2f}")


GridSearchCV runtime: 11.6s
Combinations tried: 16
Best params: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best CV RMSE: 33526.26


In [6]:
# RandomizedSearchCV — Sampled Search on Larger Grid

start = time.time()
random_search = RandomizedSearchCV(
    pipeline, param_dist_large, cv=kf,
    scoring='neg_root_mean_squared_error', n_jobs=-1,
    n_iter=30, random_state=42
)
random_search.fit(X, y)
random_time = time.time() - start

print(f"RandomizedSearchCV runtime: {random_time:.1f}s")
print(f"Combinations tried: {len(random_search.cv_results_['params'])}  "
      f"(out of {6*6*5*5*3} possible in full grid)")
print(f"Best params: {random_search.best_params_}")
print(f"Best CV RMSE: {-random_search.best_score_:.2f}")


RandomizedSearchCV runtime: 9.1s
Combinations tried: 30  (out of 2700 possible in full grid)
Best params: {'model__n_estimators': 300, 'model__min_samples_split': 8, 'model__min_samples_leaf': 1, 'model__max_features': None, 'model__max_depth': 10}
Best CV RMSE: 35278.71


In [7]:
# Comparison Table

print(f"{'Method':<25}{'RMSE':<10}{'Runtime(s)':<12}{'#Fits'}")
print(f"{'Baseline (default)':<25}{baseline_scores.mean():<10.2f}{'-':<12}{'-'}")
print(f"{'GridSearchCV':<25}{-grid_search.best_score_:<10.2f}{grid_time:<12.1f}{len(grid_search.cv_results_['params'])*5}")
print(f"{'RandomizedSearchCV':<25}{-random_search.best_score_:<10.2f}{random_time:<12.1f}{len(random_search.cv_results_['params'])*5}")


Method                   RMSE      Runtime(s)  #Fits
Baseline (default)       34531.26  -           -
GridSearchCV             33526.26  11.6        80
RandomizedSearchCV       35278.71  9.1         150
